# 01 - Data check

**EXSU 500 · MSK 2024 prostate cancer cohort**


## 1. Load processed data

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "processed").is_dir():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "processed"

clinical = pd.read_csv(DATA_DIR / "clinical_manifest.csv")
mutations = pd.read_csv(DATA_DIR / "mutations_long.csv")
cna = pd.read_csv(DATA_DIR / "cna_long.csv")
panels = pd.read_csv(DATA_DIR / "gene_panel_matrix.csv")

for name, frame in {"Clinical": clinical, "Mutations": mutations, "CNA": cna, "Panels": panels}.items():
    print(f"{name}: {frame.shape}")

Clinical: (1967, 16)
Mutations: (9651, 15)
CNA: (1064147, 4)
Panels: (1967, 5)


## 2. Verify cohort integrity and class distribution

In [2]:
assert clinical["PATIENT_ID"].is_unique
assert clinical["SAMPLE_ID"].is_unique
assert clinical["stage_binary"].isin([0, 1]).all()
assert mutations["SAMPLE_ID"].isin(clinical["SAMPLE_ID"]).all()
assert cna["SAMPLE_ID"].isin(clinical["SAMPLE_ID"]).all()
assert panels["SAMPLE_ID"].isin(clinical["SAMPLE_ID"]).all()
assert not cna.duplicated(["SAMPLE_ID", "HUGO_SYMBOL"]).any()

print("Stage distribution:")
print(clinical["stage_binary"].value_counts().sort_index())
print("\nSequencing panels:")
print(clinical["GENE_PANEL"].value_counts())
print("\nSpecimen type vs stage:")
print(pd.crosstab(clinical["SAMPLE_TYPE"], clinical["stage_binary"]))
print("\nPASS: Structural consistency checks")

Stage distribution:
stage_binary
0     943
1    1024
Name: count, dtype: int64

Sequencing panels:
GENE_PANEL
IMPACT468    1265
IMPACT410     425
IMPACT505     146
IMPACT341     131
Name: count, dtype: int64

Specimen type vs stage:
stage_binary        0    1
SAMPLE_TYPE               
Local Recurrence    0    2
Metastasis        349  378
Primary           594  643
Unknown             0    1

PASS: Structural consistency checks
